# Manual Round

You have to submit a single limit order (price, quantity). When the auction ends, the exchange selects a single clearing price that:

1. maximizes total traded volume, then
2. breaks ties by choosing the higher price.

All bids with price ≥ clearing price and asks with price ≤ clearing price execute at the clearing price. Allocation is price priority, then time priority. Since you are last to submit, you are last in line at any price level you join.

As a special welcoming event, the Intarians are organizing an exchange auction. You will have access to two stale order books: one for Dryland Flax and one for Ember Mushroom. You can put in one order per tradable good. When the auction clears, it will execute with price-time priority and there will be no more continuous trading to parttake in. How might you turn this celebratory opportunity into profit?

Note: Price-time priority is a core order-matching algorithm where exchanges fill orders based first on the best price (highest bid/lowest offer) and second on the time they were placed (FIFO - First In, First Out). This system provides fairness, as it rewards traders for posting competitive prices and being quick to fill orders.

# Dryland Flax
This order book shows a stale state of bids and asks for Dryland Flax. Use this to determine your own strategy and order input. Any Dryland Flax you acquire will be sold directly after the auction closes for **30 XIRECs** per piece.

In [ ]:
flax_bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000)
]
flax_asks = [
    (28, 40000), 
    (31, 20000), 
    (32, 20000), 
    (33, 30000)
]

mush_bids = [(20, 43000), (19, 17000), (18, 6000), (17, 5000), (16, 10000), (15, 5000), (14, 10000), (13, 7000)]
mush_asks = [(12, 20000), (13, 25000), (14, 35000), (15, 6000), (16, 5000), (18, 10000), (19, 12000)]

from dataclasses import dataclass

@dataclass
class Order:
    price: int
    qty: int
    time: int # My Time is 1, Book is 0 (book gets priority if equal prices)

def get_clearing_price(bids, asks):
    prices = sorted({o.price for o in bids + asks})
    best_vol, cp = -1, -1
    for p in prices:
        vol = min(sum(o.qty for o in bids if o.price >= p),
                  sum(o.qty for o in asks if o.price <= p))
        if vol > best_vol or (vol == best_vol and vol > 0 and p > cp):
            best_vol, cp = vol, p
    return cp

def simulate_my_pnl(my_price, my_qty, book_bids, book_asks, fv, fee):
    bids = [Order(p, q, 0) for p, q in book_bids] + [Order(my_price, my_qty, 1)] # <-- my order added to bids
    asks = [Order(p, q, 0) for p, q in book_asks]
    cp = get_clearing_price(bids, asks)

    if cp == -1 or my_price < cp:
        return 0.0, 0, cp

    supply = sum(o.qty for o in asks if o.price <= cp)
    priority = sum(o.qty for o in bids if o.price > my_price or
                   (o.price == my_price and o.time == 0))
    fill = max(0, min(my_qty, supply - priority))
    return fill * (fv - cp - fee), fill, cp

def brute_force(book_bids, book_asks, fv, fee):
    best = {"pnl": -1.0}
    prices = sorted({p for p, _ in book_bids + book_asks})

    for p in prices:
        for q in range(1, 100_001):
            pnl, fill, cp = simulate_my_pnl(p, q, book_bids, book_asks, fv, fee)
            if pnl > best["pnl"]:
                best = {"pnl": round(pnl, 2), "bid_p": p, "bid_q": q, "fill": fill, "cp": cp}
    return best


print("Flax:", brute_force(flax_bids, flax_asks, 30.0, 0.0))
print("Mushroom:", brute_force(mush_bids, mush_asks, 20.0, 0.1))

Flax: {'pnl': 9999.0, 'bid_p': 30, 'bid_q': 9999, 'fill': 9999, 'cp': 29}
Mushroom: {'pnl': 77996.1, 'bid_p': 17, 'bid_q': 19999, 'fill': 19999, 'cp': 16}


In [ ]:
bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000)
]
asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000)
]
# Normal Clear Price -> 28
# Bid 5000-> still 29
# bid 9999 @ 29-> still 29
# Clear price must be <=29 to be profitable
def clear_book(bids, asks):
    """Clear an order book by matching bids >= lowest ask at the lowest crossing price.
    Respects price-time priority (earlier entries at same price fill first)."""
    # Sort by price desc; stable sort preserves insertion order at same price
    bids = [[p, q] for p, q in sorted(bids, key=lambda x: x[0], reverse=True)]
    asks = [[p, q] for p, q in sorted(asks, key=lambda x: x[0])]

    trades = []

    for ask in asks:
        for bid in bids:
            if bid[0] < ask[0] or bid[1] == 0 or ask[1] == 0:
                continue

            fill_qty = min(bid[1], ask[1])

            trades.append({
                "price": ask[0],
                "quantity": fill_qty,
                "bid_at": bid[0],
                "ask_at": ask[0],
            })

            bid[1] -= fill_qty
            ask[1] -= fill_qty

    bids = [(p, q) for p, q in bids if q > 0]
    asks = [(p, q) for p, q in asks if q > 0]

    return bids, asks, trades


def diff(bids, asks, side, price, volume):
    """Add an order to the book, clear it, and report fill status.
    New orders are appended to the end so existing orders at the
    same price level always have time priority."""
    # Tag each entry so we can identify our order after clearing
    tag = object()

    if side == "bid":
        tagged_bids = [(p, q, None) for p, q in bids] + [(price, volume, tag)]
        tagged_asks = [(p, q, None) for p, q in asks]
    else:
        tagged_bids = [(p, q, None) for p, q in bids]
        tagged_asks = [(p, q, None) for p, q in asks] + [(price, volume, tag)]

    # Sort preserving insertion order at same price (stable sort)
    tagged_bids = sorted(tagged_bids, key=lambda x: x[0], reverse=True)
    tagged_asks = sorted(tagged_asks, key=lambda x: x[0])

    # Convert to mutable with tags
    bids_m = [[p, q, t] for p, q, t in tagged_bids]
    asks_m = [[p, q, t] for p, q, t in tagged_asks]

    trades = []

    for ask in asks_m:
        for bid in bids_m:
            if bid[0] < ask[0] or bid[1] == 0 or ask[1] == 0:
                continue

            fill_qty = min(bid[1], ask[1])

            trades.append({
                "price": ask[0],
                "quantity": fill_qty,
                "bid_at": bid[0],
                "ask_at": ask[0],
                "_bid_tag": bid[2],
                "_ask_tag": ask[2],
            })

            bid[1] -= fill_qty
            ask[1] -= fill_qty

    # Rebuild clean book
    new_bids = [(p, q) for p, q, _ in bids_m if q > 0]
    new_asks = [(p, q) for p, q, _ in asks_m if q > 0]

    # Find our fills via tag
    my_trades = [t for t in trades if t.get("_bid_tag") is tag or t.get("_ask_tag") is tag]

    # Clean internal tags from trade output
    clean_trades = [{k: v for k, v in t.items() if not k.startswith("_")} for t in my_trades]

    filled_qty = sum(t["quantity"] for t in my_trades)
    remaining = volume - filled_qty

    fill_report = {
        "side": side,
        "price": price,
        "original_volume": volume,
        "filled": filled_qty,
        "remaining": remaining,
        "fully_filled": remaining == 0,
        "trades": clean_trades,
    }

    return new_bids, new_asks, fill_report

diff(bids, asks, 'bid', 29, 11000)

([(29, 6000), (28, 12000), (27, 28000)],
 [(31, 20000), (32, 20000), (33, 30000)],
 {'side': 'bid',
  'price': 29,
  'original_volume': 11000,
  'filled': 5000,
  'remaining': 6000,
  'fully_filled': False,
  'trades': [{'price': 28, 'quantity': 5000, 'bid_at': 29, 'ask_at': 28}]})

In [ ]:

bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000)
]
asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000)
]




IndexError: list assignment index out of range

ideal: Bid price of 31 for 40000

In [87]:

bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000)
]
asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000)
]

def simulate(order, price, volume):
    

    bids.append((price, 0))
    bids.sort(reverse=True)  # Sort bids by price descending
    # PROBLEM: IT WILL SORT BY VOLUME IF THE SAME, IT SHOULD ALWAYS BE BEHIND!!!!
    bids[bids.index((price,0))] = (price,volume) # THIS FIXES THAT ISSUE! UNLESS THERE IS ALR ORDER at 0! lowk might not matter tho
    

    print(bids)
    
    cleared_bids, cleared_asks, total_cleared_volume = clear_orders(bids, asks)
    print(cleared_bids)
    if (price, volume) in cleared_bids:
         print(f"Order {order} at price {price} and volume {volume} was cleared.")

simulate('bid', 28, 12000)
#simulate('bid', 31, 5000)

[(30, 30000), (29, 5000), (28, 12000), (28, 12000), (27, 28000)]
[(30, 30000), (29, 5000), (28, 12000), (28, 12000)]
Order bid at price 28 and volume 12000 was cleared.


In [ ]:
# Strategy

# Bid for 40000 at 31 -> we get it for 28 though

# Ember Mushrooms
This order book shows a stale state of bids and asks for Ember Mushroom. Use this to determine your own strategy and order input. **Trading fees per volume apply: 0.05 XIRECs for buying and 0.05 XIRECs for selling**. Any Ember Mushroom you acquire will be sold directly after the auction closes for **20 XIRECs** per piece

In [3]:
bids = {
    20: 43000,
    19: 17000,
    18: 6000,
    17: 5000,
    16: 10000,
    15: 5000,
    14: 10000,
    13: 7000
}

asks = {
    12: 20000,
    13: 25000,
    14: 35000,
    15: 6000,
    16: 5000,
    17: 0,
    18: 10000,
    19: 12000
}